# Financial Parser Output to Structured Chunk Records

This notebook converts parsed filing JSON produced by the XBRL parser into structured chunk records for downstream vector ingestion.

Each output chunk contains:
- `chunk_id`
- `text`
- `metadata`

This notebook is responsible only for:
1. loading parsed filing JSON files
2. standardizing parser output into a chunk-ready structure
3. grouping records using filing metadata, taxonomy, contexts, and dimensions
4. generating chunk text for filing metadata, raw facts, unified facts, and narrative disclosures
5. attaching rich provenance metadata to every chunk
6. validating chunk quality and coverage
7. saving final structured chunk records and ingestion audit outputs as JSON

This notebook is designed for:
- full-document parsing output, not one hardcoded quarter
- all XML-derived JSON files from a folder
- all valid contexts and periods
- commercial / NBFC / banking / insurance filings
- preservation of taxonomy, context, dimension, and source-tag provenance

This notebook does **not** generate embeddings or upload data to Qdrant.

## 1. Import Required Libraries

In [3]:
import json
import re
import hashlib
from html import unescape
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter, defaultdict

## 2. Configure Input, Output, and Chunking Parameters

In [4]:
from pathlib import Path

BASE_DATA_FOLDER = Path("../data/processed/")

INPUT_JSON_FOLDER = BASE_DATA_FOLDER
OUTPUT_CHUNK_FOLDER = BASE_DATA_FOLDER / "chunks"
OUTPUT_AUDIT_FOLDER = BASE_DATA_FOLDER / "audits"
OUTPUT_JSON_PATH = OUTPUT_CHUNK_FOLDER / "financial_structured_chunks_v2.json"
AUDIT_JSON_PATH = OUTPUT_AUDIT_FOLDER / "financial_structured_chunks_audit_v2.json"

MAX_WORDS_PER_CHUNK = 450
INCLUDE_CONTEXT_PREFIX_IN_TEXT = True
MARKDOWN_STYLE = "bullet"
DISCLOSURE_MIN_TEXT_LENGTH = 120
DISCLOSURE_MAX_WORDS_PER_CHUNK = 350

json_files = sorted(INPUT_JSON_FOLDER.glob("*.json"))

OUTPUT_CHUNK_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_AUDIT_FOLDER.mkdir(parents=True, exist_ok=True)

print("Input folder :", INPUT_JSON_FOLDER.resolve())
print("Output file  :", OUTPUT_JSON_PATH.resolve())
print("Audit file   :", AUDIT_JSON_PATH.resolve())
print("-" * 50)
print("JSON files to process :", len(json_files))
print("Max words per chunk   :", MAX_WORDS_PER_CHUNK)
print("Disclosure max words  :", DISCLOSURE_MAX_WORDS_PER_CHUNK)

Input folder : C:\Users\Home\llmops-nse-rag\data\processed
Output file  : C:\Users\Home\llmops-nse-rag\data\processed\chunks\financial_structured_chunks_v2.json
Audit file   : C:\Users\Home\llmops-nse-rag\data\processed\audits\financial_structured_chunks_audit_v2.json
--------------------------------------------------
JSON files to process : 9
Max words per chunk   : 450
Disclosure max words  : 350


## 3. Define Section Names and Unified Field Routing

In [5]:
SECTION_DISPLAY_NAMES = {
    "filing_metadata": "Filing Metadata",
    "reporting_metadata": "Reporting Metadata & Governance",
    "income_statement_profitability": "Income Statement & Profitability",
    "capital_eps_ratios": "Capital, EPS & Ratios",
    "other_comprehensive_income": "Other Comprehensive Income (OCI)",
    "notes_disclosures": "Narrative Notes & Disclosures",
    "segment_performance": "Segment Performance",
    "cashflow_reconciliation": "Cash Flow Reconciliation Adjustments",
    "cashflow_statement": "Cash Flow Statement",
    "balance_sheet_assets": "Balance Sheet – Assets",
    "balance_sheet_equity_liabilities": "Balance Sheet – Equity & Liabilities",
    "unclassified": "Unclassified Financial Items",
    "unified_summary": "Unified Financial Summary",
}

UNIFIED_FIELD_TO_SECTION = {
    "Operating_Revenue": "income_statement_profitability",
    "Total_Income": "income_statement_profitability",
    "Primary_Direct_Expense": "income_statement_profitability",
    "Employee_Expenses": "income_statement_profitability",
    "Net_Profit": "income_statement_profitability",
    "Basic_EPS": "capital_eps_ratios",
}

ASSET_KEYWORDS = [
    "asset", "propertyplantandequipment", "capitalworkinprogress",
    "goodwill", "intangible", "investment", "inventor", "receivable",
    "loan", "cashandcashequivalent", "bankbalance", "financialasset",
    "deferredtaxasset", "heldforsale", "rightofuseasset", "capitaladvance",
]

LIABILITY_EQUITY_KEYWORDS = [
    "equity", "sharecapital", "noncontrollinginterest", "borrowings",
    "payable", "liabilit", "provision", "deferredtaxliabilit", "leaseLiabilit",
    "governmentgrant", "reserve", "surplus", "minorityinterest",
]

CASHFLOW_KEYWORDS = [
    "cashflowsfrom", "usedinoperatingactivities", "usedininvestingactivities",
    "usedinfinancingactivities", "netcashgenerated", "netcashused",
    "increaseincash", "decreaseincash", "cashandcashequivalentscashflowstatement",
    "effectofexchangeratechangesoncashandcashequivalents",
]

CASHFLOW_RECON_KEYWORDS = [
    "adjustmentsfor", "otheradjustments", "reconcileprofitloss", "depreciation",
    "amortisation", "amortization", "impairment", "gainlossondisposal",
    "baddebts", "provisionfordoubtful", "fairvaluechanges", "financecost",
]

OCI_KEYWORDS = [
    "othercomprehensiveincome", "itemthatwillnotbereclassified",
    "itemthatwillbereclassified", "remeasurement", "foreigncurrencytranslationreserve",
]

RATIO_KEYWORDS = [
    "earningslosspershare", "ratio", "facevalueofequitysharecapital",
    "paidupvalueofequitysharecapital", "reserveexcludingrevaluationreserves",
    "capitaladequacy", "crar", "tieri", "tierii", "bookvaluepershare",
]

METADATA_KEYWORDS = [
    "nameofthecompany", "nameofbank", "nameofentity", "symbol", "scripcode",
    "isin", "descriptionofpresentationcurrency", "levelofrounding",
    "natureofreportstandaloneconsolidated", "whether", "dateof", "timeof",
    "reportingquarter", "declaration", "validitydate",
]

DISCLOSURE_KEYWORDS = [
    "textblock", "disclosure", "policy", "note", "narrative",
    "accountingpolicy", "riskmanagement", "maturityanalysis",
]

print("Section configuration ready.")

Section configuration ready.


## 4. Define Base File and Text Utility Functions

In [6]:
def load_json(json_path: Path) -> Any:
    path = Path(json_path)
    if not path.exists():
        raise FileNotFoundError(f"Input JSON file not found: {path.resolve()}")
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def save_json(data: Any, output_path: Path) -> Path:
    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
    return path


def normalize_key(text: Any) -> str:
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def slugify(text: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "-", str(text).lower()).strip("-")


def short_hash(text: Any, length: int = 10) -> str:
    return hashlib.md5(str(text).encode("utf-8")).hexdigest()[:length]


def is_null_or_blank(value: Any) -> bool:
    return value is None or (isinstance(value, str) and value.strip() == "")


def count_words(text: Any) -> int:
    return len(re.findall(r"\b\w+\b", str(text)))


def get_source_file_name(path: Path) -> str:
    return Path(path).name if path else "unknown_source.json"


def ensure_list(value: Any) -> List[Any]:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]


print("Base utilities ready.")

Base utilities ready.


## 5. Define Text Cleaning and Numeric Formatting Helpers

In [7]:
def clean_text_value(value: Any) -> str:
    if is_null_or_blank(value):
        return "Not reported"

    text = unescape(str(value)).strip()
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"</p\s*>", "\n\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<li\s*>", "\n- ", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("&nbsp;", " ")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)

    lowered = text.lower().strip()
    if lowered == "true":
        return "Yes"
    if lowered == "false":
        return "No"

    return text.strip()


def is_numeric_like(value: Any) -> bool:
    if isinstance(value, (int, float)):
        return True
    if not isinstance(value, str):
        return False
    cleaned = value.replace(",", "").strip()
    return bool(re.fullmatch(r"-?\d+(\.\d+)?", cleaned))


def format_plain_number(number: float, decimals: int = 4) -> str:
    if float(number).is_integer():
        return f"{int(number):,}"
    return f"{number:,.{decimals}f}".rstrip("0").rstrip(".")


def extract_display_value_from_fact(fact: Dict[str, Any], currency: Optional[str] = None) -> str:
    normalized_value = fact.get("normalized_value")
    raw_value = fact.get("raw_value")
    text_value = fact.get("text_value")
    is_numeric = fact.get("is_numeric")

    if is_numeric is True:
        candidate = normalized_value if normalized_value is not None else raw_value
        if candidate is None:
            return "Not reported"
        try:
            numeric_value = float(str(candidate).replace(",", "").strip())
            prefix = f"{currency} " if currency and currency != "Unknown Currency" else ""
            return f"{prefix}{format_plain_number(numeric_value, 4)}"
        except Exception:
            return clean_text_value(candidate)

    if not is_null_or_blank(text_value):
        return clean_text_value(text_value)

    if not is_null_or_blank(raw_value):
        return clean_text_value(raw_value)

    return "Not reported"


print("Text and numeric helpers ready.")

Text and numeric helpers ready.


## 6. Define Parser Output Access Helpers

In [51]:
def first_present(mapping: Dict[str, Any], candidates: List[str], default: Any = None) -> Any:
    for key in candidates:
        if key in mapping and mapping.get(key) is not None:
            return mapping.get(key)
    return default


def extract_filing_metadata(parsed_json: Dict[str, Any]) -> Dict[str, Any]:
    filing_metadata = first_present(
        parsed_json,
        ["filing_metadata", "metadata", "file_metadata", "document_metadata"],
        default={},
    )
    if not isinstance(filing_metadata, dict):
        filing_metadata = {}

    company_name = first_present(
        filing_metadata,
        ["company_name", "NameOfTheCompany", "NameOfBank", "NameOfEntity"],
        default="Unknown Company",
    )
    currency = first_present(
        filing_metadata,
        ["currency", "DescriptionOfPresentationCurrency"],
        default="Unknown Currency",
    )
    rounding = first_present(
        filing_metadata,
        ["rounding", "LevelOfRounding"],
        default="Unknown Rounding",
    )
    report_type = first_present(
        filing_metadata,
        ["report_type", "NatureOfReportStandaloneConsolidated"],
        default="Unknown Report Type",
    )

    return {
        "company_name": company_name,
        "currency": currency,
        "rounding": rounding,
        "report_type": report_type,
        "raw_filing_metadata": filing_metadata,
    }


def extract_taxonomy(parsed_json: Dict[str, Any]) -> str:
    return first_present(parsed_json, ["taxonomy", "detected_taxonomy"], default="UNKNOWN")


def extract_contexts(parsed_json: Dict[str, Any]) -> Dict[str, Any]:
    contexts = first_present(parsed_json, ["contexts", "context_dict", "context_map"], default={})
    return contexts if isinstance(contexts, dict) else {}


def extract_raw_facts(parsed_json: Dict[str, Any]) -> List[Dict[str, Any]]:
    candidates = [
        "raw_facts", "raw_fact_records", "facts", "fact_records", "all_facts",
    ]
    for key in candidates:
        value = parsed_json.get(key)
        if isinstance(value, list):
            return [item for item in value if isinstance(item, dict)]
    return []


def extract_unified_facts(parsed_json: Dict[str, Any]) -> List[Dict[str, Any]]:
    candidates = [
        "unified_facts", "unified_records", "mapped_facts", "unified_mapped_records",
    ]
    for key in candidates:
        value = parsed_json.get(key)
        if isinstance(value, list):
            return [item for item in value if isinstance(item, dict)]
    return []


def extract_source_file(parsed_json: Dict[str, Any], fallback_path: Path) -> str:
    return first_present(parsed_json, ["source_file", "file_name", "xml_file"], default=get_source_file_name(fallback_path))


print("Parser output access helpers ready.")

Parser output access helpers ready.


## 7. Inspect Sample Parser Output Contracts

In [52]:
sample_contracts = []

for sample_path in json_files[:3]:
    try:
        parsed = load_json(sample_path)
        if not isinstance(parsed, dict):
            sample_contracts.append({
                "file": sample_path.name,
                "top_level_type": type(parsed).__name__,
                "top_level_keys": [],
                "raw_facts_count": 0,
                "unified_facts_count": 0,
                "contexts_count": 0,
            })
            continue

        sample_contracts.append({
            "file": sample_path.name,
            "top_level_type": type(parsed).__name__,
            "top_level_keys": sorted(parsed.keys()),
            "raw_facts_count": len(extract_raw_facts(parsed)),
            "unified_facts_count": len(extract_unified_facts(parsed)),
            "contexts_count": len(extract_contexts(parsed)),
            "taxonomy": extract_taxonomy(parsed),
            "filing_metadata_keys": sorted(extract_filing_metadata(parsed)["raw_filing_metadata"].keys()),
        })
    except Exception as exc:
        sample_contracts.append({
            "file": sample_path.name,
            "error": str(exc),
        })

print(json.dumps(sample_contracts, indent=2, ensure_ascii=False))

[
  {
    "file": "GI_117338_1350247_17012025074725.json",
    "top_level_type": "dict",
    "top_level_keys": [
      "context_count",
      "contexts",
      "filing_metadata",
      "raw_fact_count",
      "raw_facts",
      "source_file",
      "taxonomy",
      "unified_record_count",
      "unified_records"
    ],
    "raw_facts_count": 339,
    "unified_facts_count": 120,
    "contexts_count": 128,
    "taxonomy": "INSURANCE",
    "filing_metadata_keys": [
      "company_name",
      "currency",
      "report_type",
      "rounding"
    ]
  },
  {
    "file": "INDAS_118123_1365717_30012025030002.json",
    "top_level_type": "dict",
    "top_level_keys": [
      "context_count",
      "contexts",
      "filing_metadata",
      "raw_fact_count",
      "raw_facts",
      "source_file",
      "taxonomy",
      "unified_record_count",
      "unified_records"
    ],
    "raw_facts_count": 129,
    "unified_facts_count": 10,
    "contexts_count": 11,
    "taxonomy": "COMMERCIAL_NBFC",


## 8. Define Context and Period Resolution Helpers

In [53]:
def resolve_context_type(context_meta: Dict[str, Any]) -> str:
    explicit_type = first_present(context_meta, ["context_type", "period_type"], default=None)
    if explicit_type:
        explicit_norm = str(explicit_type).strip().lower()
        if explicit_norm in {"duration", "instant"}:
            return explicit_norm.title()

    if context_meta.get("instant_date") or context_meta.get("instant"):
        return "Instant"

    if context_meta.get("start_date") or context_meta.get("end_date") or context_meta.get("startDate") or context_meta.get("endDate"):
        return "Duration"

    return "Unknown"


def resolve_context_dates(context_meta: Dict[str, Any]) -> Dict[str, Optional[str]]:
    start_date = first_present(context_meta, ["start_date", "startDate"], default=None)
    end_date = first_present(context_meta, ["end_date", "endDate"], default=None)
    instant_date = first_present(context_meta, ["instant_date", "instant"], default=None)

    return {
        "start_date": start_date,
        "end_date": end_date,
        "instant_date": instant_date,
    }


def resolve_dimension_members(fact: Dict[str, Any], context_meta: Dict[str, Any]) -> List[Dict[str, str]]:
    fact_dims = fact.get("dimensions")
    context_dimension = context_meta.get("dimension")
    context_member = context_meta.get("member")

    if isinstance(fact_dims, list):
        output = []
        for item in fact_dims:
            if isinstance(item, dict):
                output.append({
                    "dimension": str(item.get("dimension", "")).strip(),
                    "member": str(item.get("member", "")).strip(),
                })
        if output:
            return output

    if isinstance(fact_dims, dict):
        return [
            {
                "dimension": str(key).strip(),
                "member": str(value).strip(),
            }
            for key, value in fact_dims.items()
        ]

    if context_dimension or context_member:
        return [{
            "dimension": str(context_dimension or "").strip(),
            "member": str(context_member or "").strip(),
        }]

    return []


def dimension_member_signature(dimensions: List[Dict[str, str]]) -> str:
    if not dimensions:
        return "entity-level"
    parts = []
    for item in dimensions:
        dim = str(item.get("dimension", "")).strip() or "unknown-dimension"
        mem = str(item.get("member", "")).strip() or "unknown-member"
        parts.append(f"{dim}={mem}")
    return " | ".join(sorted(parts))


def build_reporting_period_label(context_type: str, dates: Dict[str, Optional[str]]) -> str:
    if context_type == "Instant" and dates.get("instant_date"):
        return f"As of {dates['instant_date']}"
    if context_type == "Duration" and dates.get("start_date") and dates.get("end_date"):
        return f"{dates['start_date']} to {dates['end_date']}"
    if dates.get("end_date"):
        return f"As of {dates['end_date']}"
    if dates.get("instant_date"):
        return f"As of {dates['instant_date']}"
    return "Unknown Period"


print("Context helpers ready.")

Context helpers ready.


## 9. Define Fact Classification Helpers

In [41]:
def is_disclosure_fact(fact: Dict[str, Any]) -> bool:
    tag_name = normalize_key(fact.get("tag_name", ""))
    text_value = clean_text_value(fact.get("text_value", ""))
    if any(keyword in tag_name for keyword in DISCLOSURE_KEYWORDS):
        return True
    if len(text_value) >= DISCLOSURE_MIN_TEXT_LENGTH and not fact.get("is_numeric", False):
        return True
    return False


def classify_raw_fact_section(fact: Dict[str, Any], context_type: str) -> str:
    tag_name = normalize_key(fact.get("tag_name", ""))
    dimensions = dimension_member_signature(resolve_dimension_members(fact, {})).lower()

    if is_disclosure_fact(fact):
        return "notes_disclosures"

    if any(keyword in tag_name for keyword in METADATA_KEYWORDS):
        return "reporting_metadata"

    if "segment" in tag_name or "segment" in dimensions or "unallocable" in tag_name:
        return "segment_performance"

    if any(keyword in tag_name for keyword in OCI_KEYWORDS):
        return "other_comprehensive_income"

    if any(keyword in tag_name for keyword in CASHFLOW_KEYWORDS):
        return "cashflow_statement"

    if any(keyword in tag_name for keyword in CASHFLOW_RECON_KEYWORDS):
        return "cashflow_reconciliation"

    if any(keyword in tag_name for keyword in RATIO_KEYWORDS):
        return "capital_eps_ratios"

    if context_type == "Instant":
        if any(keyword in tag_name for keyword in ASSET_KEYWORDS):
            return "balance_sheet_assets"
        if any(keyword in tag_name for keyword in LIABILITY_EQUITY_KEYWORDS):
            return "balance_sheet_equity_liabilities"

    if context_type == "Instant":
        return "balance_sheet_assets"

    if tag_name in {"profitlossforperiod", "profitlossfortheperiod", "profitlossaftertax"}:
        return "income_statement_profitability"

    return "income_statement_profitability"


def classify_unified_fact_section(unified_fact: Dict[str, Any], fallback_context_type: str) -> str:
    unified_field = unified_fact.get("unified_field")
    if unified_field in UNIFIED_FIELD_TO_SECTION:
        return UNIFIED_FIELD_TO_SECTION[unified_field]
    if fallback_context_type == "Instant":
        return "balance_sheet_assets"
    return "unified_summary"


print("Classification helpers ready.")

Classification helpers ready.


## 10. Define Parser Output Standardization Helpers

In [42]:
def standardize_raw_fact(
    fact: Dict[str, Any],
    taxonomy: str,
    filing_metadata: Dict[str, Any],
    contexts: Dict[str, Any],
) -> Dict[str, Any]:
    context_id = first_present(fact, ["context_id", "contextRef", "context_ref"], default=None)
    context_meta = contexts.get(context_id, {}) if context_id else {}
    context_type = resolve_context_type(context_meta)
    dates = resolve_context_dates(context_meta)
    dimensions = resolve_dimension_members(fact, context_meta)
    section_key = classify_raw_fact_section(fact, context_type)
    reporting_period = build_reporting_period_label(context_type, dates)

    standardized = {
        "record_type": "raw_fact",
        "taxonomy": taxonomy,
        "company_name": filing_metadata["company_name"],
        "currency": filing_metadata["currency"],
        "rounding": filing_metadata["rounding"],
        "report_type": filing_metadata["report_type"],
        "context_id": context_id or "no_context",
        "context_type": context_type,
        "start_date": dates["start_date"],
        "end_date": dates["end_date"],
        "instant_date": dates["instant_date"],
        "reporting_period": reporting_period,
        "dimensions": dimensions,
        "dimension_member_signature": dimension_member_signature(dimensions),
        "tag_name": first_present(fact, ["tag_name", "name", "fact_name"], default="UnknownTag"),
        "text_value": first_present(fact, ["text_value", "value", "text"], default=None),
        "raw_value": first_present(fact, ["raw_value", "value"], default=None),
        "decimals": fact.get("decimals"),
        "normalized_value": fact.get("normalized_value"),
        "is_numeric": fact.get("is_numeric", False),
        "section_key": section_key,
        "source_tag": first_present(fact, ["tag_name", "name", "fact_name"], default="UnknownTag"),
        "unified_field": None,
        "raw_record": fact,
    }
    return standardized


def standardize_unified_fact(
    unified_fact: Dict[str, Any],
    taxonomy: str,
    filing_metadata: Dict[str, Any],
    contexts: Dict[str, Any],
) -> Dict[str, Any]:
    context_id = first_present(unified_fact, ["context_id", "contextRef", "context_ref"], default=None)
    context_meta = contexts.get(context_id, {}) if context_id else {}
    context_type = resolve_context_type(context_meta)
    dates = resolve_context_dates(context_meta)
    dimensions = resolve_dimension_members(unified_fact, context_meta)
    reporting_period = build_reporting_period_label(context_type, dates)

    source_tag = first_present(
        unified_fact,
        ["source_tag", "source_tag_name", "tag_name"],
        default="UnknownSourceTag",
    )
    unified_field = first_present(
        unified_fact,
        ["unified_field", "mapped_field", "field_name"],
        default="UnknownUnifiedField",
    )
    normalized_value = first_present(unified_fact, ["normalized_value", "value"], default=None)
    raw_value = first_present(unified_fact, ["raw_value", "value"], default=None)

    standardized = {
        "record_type": "unified_fact",
        "taxonomy": taxonomy,
        "company_name": filing_metadata["company_name"],
        "currency": filing_metadata["currency"],
        "rounding": filing_metadata["rounding"],
        "report_type": filing_metadata["report_type"],
        "context_id": context_id or "no_context",
        "context_type": context_type,
        "start_date": dates["start_date"],
        "end_date": dates["end_date"],
        "instant_date": dates["instant_date"],
        "reporting_period": reporting_period,
        "dimensions": dimensions,
        "dimension_member_signature": dimension_member_signature(dimensions),
        "tag_name": source_tag,
        "text_value": unified_fact.get("text_value"),
        "raw_value": raw_value,
        "decimals": unified_fact.get("decimals"),
        "normalized_value": normalized_value,
        "is_numeric": unified_fact.get("is_numeric", True),
        "section_key": classify_unified_fact_section(unified_fact, context_type),
        "source_tag": source_tag,
        "unified_field": unified_field,
        "raw_record": unified_fact,
    }
    return standardized


def standardize_filing(parsed_json: Dict[str, Any], file_path: Path) -> Dict[str, Any]:
    taxonomy = extract_taxonomy(parsed_json)
    filing_metadata = extract_filing_metadata(parsed_json)
    contexts = extract_contexts(parsed_json)
    raw_facts = extract_raw_facts(parsed_json)
    unified_facts = extract_unified_facts(parsed_json)
    source_file = extract_source_file(parsed_json, file_path)

    standardized_raw = [
        standardize_raw_fact(fact, taxonomy, filing_metadata, contexts)
        for fact in raw_facts
    ]
    standardized_unified = [
        standardize_unified_fact(fact, taxonomy, filing_metadata, contexts)
        for fact in unified_facts
    ]

    return {
        "source_file": source_file,
        "taxonomy": taxonomy,
        "filing_metadata": filing_metadata,
        "contexts": contexts,
        "standardized_raw_facts": standardized_raw,
        "standardized_unified_facts": standardized_unified,
    }


print("Standardization helpers ready.")

Standardization helpers ready.


## 11. Preview Standardized Filing Structure

In [43]:
if json_files:
    sample_parsed = load_json(json_files[0])
    standardized_preview = standardize_filing(sample_parsed, json_files[0])

    print("Source file:", standardized_preview["source_file"])
    print("Taxonomy   :", standardized_preview["taxonomy"])
    print("Company    :", standardized_preview["filing_metadata"]["company_name"])
    print("Raw facts  :", len(standardized_preview["standardized_raw_facts"]))
    print("Unified    :", len(standardized_preview["standardized_unified_facts"]))
    print("Contexts   :", len(standardized_preview["contexts"]))

    if standardized_preview["standardized_raw_facts"]:
        print("\nSample standardized raw fact:")
        print(json.dumps(standardized_preview["standardized_raw_facts"][0], indent=2, ensure_ascii=False, default=str))

    if standardized_preview["standardized_unified_facts"]:
        print("\nSample standardized unified fact:")
        print(json.dumps(standardized_preview["standardized_unified_facts"][0], indent=2, ensure_ascii=False, default=str))
else:
    print("No JSON files found.")

Source file: GI_117338_1350247_17012025074725.xml
Taxonomy   : INSURANCE
Company    : ICICI Lombard General Insurance Company Limited
Raw facts  : 339
Unified    : 120
Contexts   : 128

Sample standardized raw fact:
{
  "record_type": "raw_fact",
  "taxonomy": "INSURANCE",
  "company_name": "ICICI Lombard General Insurance Company Limited",
  "currency": "INR",
  "rounding": "Unknown Rounding",
  "report_type": "Standalone",
  "context_id": "OneI",
  "context_type": "Instant",
  "start_date": null,
  "end_date": null,
  "instant_date": "2024-12-31",
  "reporting_period": "As of 2024-12-31",
  "dimensions": [],
  "dimension_member_signature": "entity-level",
  "tag_name": "ScripCode",
  "text_value": "540716",
  "raw_value": "540716",
  "decimals": null,
  "normalized_value": 540716.0,
  "is_numeric": true,
  "section_key": "reporting_metadata",
  "source_tag": "ScripCode",
  "unified_field": null,
  "raw_record": {
    "tag_name": "ScripCode",
    "text_value": "540716",
    "raw_value

## 12. Define Rendering Helpers for Metadata and Facts

In [14]:
def prettify_label(raw_key: Any) -> str:
    label = str(raw_key).replace("_", " ")
    label = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", label)
    label = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", " ", label)
    return re.sub(r"\s+", " ", label).strip()


def render_filing_metadata_lines(
    source_file: str,
    taxonomy: str,
    filing_metadata: Dict[str, Any],
) -> List[str]:
    return [
        f"- Company Name: {clean_text_value(filing_metadata.get('company_name'))}",
        f"- Taxonomy: {clean_text_value(taxonomy)}",
        f"- Report Type: {clean_text_value(filing_metadata.get('report_type'))}",
        f"- Presentation Currency: {clean_text_value(filing_metadata.get('currency'))}",
        f"- Rounding: {clean_text_value(filing_metadata.get('rounding'))}",
        f"- Source File: {clean_text_value(source_file)}",
    ]


def render_raw_fact_line(fact: Dict[str, Any]) -> str:
    label = prettify_label(fact.get("tag_name"))
    value = extract_display_value_from_fact(fact, currency=fact.get("currency"))
    return f"- {label}: {value}"


def render_unified_fact_line(fact: Dict[str, Any]) -> str:
    unified_field = fact.get("unified_field") or "Unknown Unified Field"
    source_tag = fact.get("source_tag") or "Unknown Source Tag"
    value = extract_display_value_from_fact(fact, currency=fact.get("currency"))
    return f"- {prettify_label(unified_field)}: {value} (source tag: {prettify_label(source_tag)})"


def split_long_text_into_paragraphs(text: str) -> List[str]:
    cleaned = clean_text_value(text)
    blocks = [block.strip() for block in re.split(r"\n\s*\n", cleaned) if block.strip()]
    if blocks:
        return blocks
    sentences = re.split(r"(?<=[.!?])\s+", cleaned)
    chunks = []
    current = []

    for sentence in sentences:
        current.append(sentence)
        if count_words(" ".join(current)) >= 80:
            chunks.append(" ".join(current).strip())
            current = []
    if current:
        chunks.append(" ".join(current).strip())

    return [item for item in chunks if item.strip()]


print("Rendering helpers ready.")

Rendering helpers ready.


## 13. Define Chunk ID and Context Prefix Helpers

In [15]:
def build_context_prefix(metadata: Dict[str, Any]) -> str:
    if not INCLUDE_CONTEXT_PREFIX_IN_TEXT:
        return ""

    prefix_lines = [
        f"Company Name: {metadata.get('company_name', 'Unknown Company')}",
        f"Taxonomy: {metadata.get('taxonomy', 'UNKNOWN')}",
        f"Report Type: {metadata.get('report_type', 'Unknown Report Type')}",
        f"Reporting Period: {metadata.get('reporting_period', 'Unknown Period')}",
        f"Context Type: {metadata.get('context_type', 'Unknown')}",
        f"Section: {metadata.get('section_name', 'Unknown Section')}",
    ]

    dimension_members = metadata.get("dimension_members", [])
    if dimension_members:
        prefix_lines.append("Dimensions: " + "; ".join(dimension_members))

    prefix_lines.append("")
    return "\n".join(prefix_lines)


def build_chunk_id(metadata: Dict[str, Any], chunk_index: int) -> str:
    tags_signature = ",".join(sorted(metadata.get("source_tags", [])))
    
    base = "|".join(
        [
            str(metadata.get("source_file", "unknown_source")),
            str(metadata.get("chunk_type", "unknown_type")),
            str(metadata.get("section_key", "unknown_section")),
            str(metadata.get("reporting_period", "unknown_period")),
            str(metadata.get("context_signature", "unknown_context")),
            tags_signature,
            str(chunk_index),
        ]
    )
    return slugify(
        f"{Path(str(metadata.get('source_file', 'unknown_source'))).stem}-{short_hash(base, 12)}"
    )


print("Chunk ID and prefix helpers ready.")

Chunk ID and prefix helpers ready.


## 14. Define Generic Chunk Splitting Helper

In [16]:
def chunk_lines_into_records(
    lines: List[str],
    metadata_base: Dict[str, Any],
    max_words_per_chunk: int,
) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    context_prefix = build_context_prefix(metadata_base)
    prefix_word_count = count_words(context_prefix)

    current_lines: List[str] = []
    current_word_count = prefix_word_count
    chunk_index = 1

    for line in lines:
        line_word_count = count_words(line)

        if current_lines and (current_word_count + line_word_count > max_words_per_chunk):
            chunk_text = (context_prefix + "\n".join(current_lines)).strip() if context_prefix else "\n".join(current_lines).strip()
            chunk_metadata = {
                **metadata_base,
                "chunk_index": chunk_index,
                "word_count": count_words(chunk_text),
            }
            records.append({
                "chunk_id": build_chunk_id(chunk_metadata, chunk_index),
                "text": chunk_text,
                "metadata": chunk_metadata,
            })
            chunk_index += 1
            current_lines = []
            current_word_count = prefix_word_count

        current_lines.append(line)
        current_word_count += line_word_count

    if current_lines:
        chunk_text = (context_prefix + "\n".join(current_lines)).strip() if context_prefix else "\n".join(current_lines).strip()
        chunk_metadata = {
            **metadata_base,
            "chunk_index": chunk_index,
            "word_count": count_words(chunk_text),
        }
        records.append({
            "chunk_id": build_chunk_id(chunk_metadata, chunk_index),
            "text": chunk_text,
            "metadata": chunk_metadata,
        })

    return records


print("Generic chunk splitter ready.")

Generic chunk splitter ready.


## 15. Define Grouping Helpers for Raw, Unified, and Disclosure Records

In [17]:
def context_signature_from_record(record: Dict[str, Any]) -> str:
    return "|".join(
        [
            str(record.get("context_id", "no_context")),
            str(record.get("context_type", "Unknown")),
            str(record.get("start_date") or ""),
            str(record.get("end_date") or ""),
            str(record.get("instant_date") or ""),
            str(record.get("dimension_member_signature") or "entity-level"),
        ]
    )


def group_raw_fact_records(records: List[Dict[str, Any]]) -> Dict[Tuple[str, str], List[Dict[str, Any]]]:
    grouped: Dict[Tuple[str, str], List[Dict[str, Any]]] = defaultdict(list)
    for record in records:
        if is_disclosure_fact(record):
            continue
        key = (record["section_key"], context_signature_from_record(record))
        grouped[key].append(record)
    return grouped


def group_unified_fact_records(records: List[Dict[str, Any]]) -> Dict[Tuple[str, str], List[Dict[str, Any]]]:
    grouped: Dict[Tuple[str, str], List[Dict[str, Any]]] = defaultdict(list)
    for record in records:
        key = (record["section_key"], context_signature_from_record(record))
        grouped[key].append(record)
    return grouped


def group_disclosure_records(records: List[Dict[str, Any]]) -> Dict[Tuple[str, str, str], List[Dict[str, Any]]]:
    grouped: Dict[Tuple[str, str, str], List[Dict[str, Any]]] = defaultdict(list)
    for record in records:
        if not is_disclosure_fact(record):
            continue
        key = (
            record["section_key"],
            context_signature_from_record(record),
            str(record.get("tag_name", "UnknownTag")),
        )
        grouped[key].append(record)
    return grouped


print("Grouping helpers ready.")

Grouping helpers ready.


## 16. Define Metadata Builder for Chunk Records

In [18]:
def build_metadata_base(
    source_file: str,
    taxonomy: str,
    filing_metadata: Dict[str, Any],
    section_key: str,
    chunk_type: str,
    reporting_period: str = "Unknown Period",
    context_type: str = "Unknown",
    context_ids: Optional[List[str]] = None,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    instant_date: Optional[str] = None,
    dimensions: Optional[List[Dict[str, str]]] = None,
    source_tags: Optional[List[str]] = None,
    unified_fields: Optional[List[str]] = None,
    context_signature: Optional[str] = None,
) -> Dict[str, Any]:
    dimensions = dimensions or []
    dimension_members = [
        f"{item.get('dimension', '')}={item.get('member', '')}".strip("=")
        for item in dimensions
        if item.get("dimension") or item.get("member")
    ]

    return {
        "company_name": filing_metadata["company_name"],
        "currency": filing_metadata["currency"],
        "rounding": filing_metadata["rounding"],
        "report_type": filing_metadata["report_type"],
        "source_file": source_file,
        "taxonomy": taxonomy,
        "section_key": section_key,
        "section_name": SECTION_DISPLAY_NAMES.get(section_key, prettify_label(section_key)),
        "chunk_type": chunk_type,
        "reporting_period": reporting_period,
        "context_type": context_type,
        "context_ids": sorted(set(context_ids or [])),
        "start_date": start_date,
        "end_date": end_date,
        "instant_date": instant_date,
        "dimension_members": sorted(set(dimension_members)),
        "source_tags": sorted(set(source_tags or [])),
        "unified_fields": sorted(set(unified_fields or [])),
        "context_signature": context_signature or "unknown_context",
        "markdown_style": MARKDOWN_STYLE,
    }


print("Metadata builder ready.")

Metadata builder ready.


## 17. Define Filing Metadata Chunk Generator

In [19]:
def generate_filing_metadata_chunks(standardized_filing: Dict[str, Any]) -> List[Dict[str, Any]]:
    source_file = standardized_filing["source_file"]
    taxonomy = standardized_filing["taxonomy"]
    filing_metadata = standardized_filing["filing_metadata"]

    metadata_base = build_metadata_base(
        source_file=source_file,
        taxonomy=taxonomy,
        filing_metadata=filing_metadata,
        section_key="filing_metadata",
        chunk_type="filing_metadata",
        reporting_period="Full Filing",
        context_type="Document",
        context_ids=[],
        context_signature="document-level",
    )

    lines = render_filing_metadata_lines(
        source_file=source_file,
        taxonomy=taxonomy,
        filing_metadata=filing_metadata,
    )

    return chunk_lines_into_records(
        lines=lines,
        metadata_base=metadata_base,
        max_words_per_chunk=MAX_WORDS_PER_CHUNK,
    )


print("Filing metadata chunk generator ready.")

Filing metadata chunk generator ready.


## 18. Define Raw Fact Chunk Generator

In [20]:
def generate_raw_fact_chunks(standardized_filing: Dict[str, Any]) -> List[Dict[str, Any]]:
    source_file = standardized_filing["source_file"]
    taxonomy = standardized_filing["taxonomy"]
    filing_metadata = standardized_filing["filing_metadata"]
    raw_records = standardized_filing["standardized_raw_facts"]

    grouped = group_raw_fact_records(raw_records)
    chunks: List[Dict[str, Any]] = []

    for (section_key, context_signature), records in grouped.items():
        sorted_records = sorted(
            records,
            key=lambda item: (
                str(item.get("tag_name", "")),
                str(item.get("dimension_member_signature", "")),
                str(item.get("context_id", "")),
            ),
        )

        first_record = sorted_records[0]
        metadata_base = build_metadata_base(
            source_file=source_file,
            taxonomy=taxonomy,
            filing_metadata=filing_metadata,
            section_key=section_key,
            chunk_type="raw_fact_group",
            reporting_period=first_record.get("reporting_period", "Unknown Period"),
            context_type=first_record.get("context_type", "Unknown"),
            context_ids=[item.get("context_id", "no_context") for item in sorted_records],
            start_date=first_record.get("start_date"),
            end_date=first_record.get("end_date"),
            instant_date=first_record.get("instant_date"),
            dimensions=first_record.get("dimensions", []),
            source_tags=[item.get("source_tag", "UnknownTag") for item in sorted_records],
            context_signature=context_signature,
        )

        lines = [render_raw_fact_line(record) for record in sorted_records]
        chunks.extend(
            chunk_lines_into_records(
                lines=lines,
                metadata_base=metadata_base,
                max_words_per_chunk=MAX_WORDS_PER_CHUNK,
            )
        )

    return chunks


print("Raw fact chunk generator ready.")

Raw fact chunk generator ready.


## 19. Define Unified Fact Chunk Generator

In [21]:
def generate_unified_fact_chunks(standardized_filing: Dict[str, Any]) -> List[Dict[str, Any]]:
    source_file = standardized_filing["source_file"]
    taxonomy = standardized_filing["taxonomy"]
    filing_metadata = standardized_filing["filing_metadata"]
    unified_records = standardized_filing["standardized_unified_facts"]

    grouped = group_unified_fact_records(unified_records)
    chunks: List[Dict[str, Any]] = []

    for (section_key, context_signature), records in grouped.items():
        sorted_records = sorted(
            records,
            key=lambda item: (
                str(item.get("unified_field", "")),
                str(item.get("source_tag", "")),
                str(item.get("context_id", "")),
            ),
        )

        first_record = sorted_records[0]
        metadata_base = build_metadata_base(
            source_file=source_file,
            taxonomy=taxonomy,
            filing_metadata=filing_metadata,
            section_key=section_key,
            chunk_type="unified_metric_group",
            reporting_period=first_record.get("reporting_period", "Unknown Period"),
            context_type=first_record.get("context_type", "Unknown"),
            context_ids=[item.get("context_id", "no_context") for item in sorted_records],
            start_date=first_record.get("start_date"),
            end_date=first_record.get("end_date"),
            instant_date=first_record.get("instant_date"),
            dimensions=first_record.get("dimensions", []),
            source_tags=[item.get("source_tag", "UnknownTag") for item in sorted_records],
            unified_fields=[item.get("unified_field", "UnknownUnifiedField") for item in sorted_records],
            context_signature=context_signature,
        )

        lines = [render_unified_fact_line(record) for record in sorted_records]
        chunks.extend(
            chunk_lines_into_records(
                lines=lines,
                metadata_base=metadata_base,
                max_words_per_chunk=MAX_WORDS_PER_CHUNK,
            )
        )

    return chunks


print("Unified fact chunk generator ready.")

Unified fact chunk generator ready.


## 20. Define Narrative Disclosure Chunk Generator

In [22]:
def generate_disclosure_chunks(standardized_filing: Dict[str, Any]) -> List[Dict[str, Any]]:
    source_file = standardized_filing["source_file"]
    taxonomy = standardized_filing["taxonomy"]
    filing_metadata = standardized_filing["filing_metadata"]
    raw_records = standardized_filing["standardized_raw_facts"]

    grouped = group_disclosure_records(raw_records)
    chunks: List[Dict[str, Any]] = []

    for (section_key, context_signature, tag_name), records in grouped.items():
        first_record = records[0]
        metadata_base = build_metadata_base(
            source_file=source_file,
            taxonomy=taxonomy,
            filing_metadata=filing_metadata,
            section_key=section_key,
            chunk_type="narrative_disclosure",
            reporting_period=first_record.get("reporting_period", "Unknown Period"),
            context_type=first_record.get("context_type", "Unknown"),
            context_ids=[item.get("context_id", "no_context") for item in records],
            start_date=first_record.get("start_date"),
            end_date=first_record.get("end_date"),
            instant_date=first_record.get("instant_date"),
            dimensions=first_record.get("dimensions", []),
            source_tags=[tag_name],
            context_signature=context_signature,
        )

        lines: List[str] = [f"### {prettify_label(tag_name)}"]
        for record in records:
            text_value = clean_text_value(record.get("text_value"))
            paragraphs = split_long_text_into_paragraphs(text_value)
            for para in paragraphs:
                lines.append(f"- {para}")

        chunks.extend(
            chunk_lines_into_records(
                lines=lines,
                metadata_base=metadata_base,
                max_words_per_chunk=DISCLOSURE_MAX_WORDS_PER_CHUNK,
            )
        )

    return chunks


print("Narrative disclosure chunk generator ready.")

Narrative disclosure chunk generator ready.


## 21. Define End-to-End Per-Filing Chunk Builder

In [23]:
def generate_chunks_for_filing(parsed_json: Dict[str, Any], file_path: Path) -> Dict[str, Any]:
    standardized_filing = standardize_filing(parsed_json, file_path)

    filing_metadata_chunks = generate_filing_metadata_chunks(standardized_filing)
    raw_fact_chunks = generate_raw_fact_chunks(standardized_filing)
    unified_fact_chunks = generate_unified_fact_chunks(standardized_filing)
    disclosure_chunks = generate_disclosure_chunks(standardized_filing)

    all_chunks = filing_metadata_chunks + raw_fact_chunks + unified_fact_chunks + disclosure_chunks

    filing_audit = {
        "source_file": standardized_filing["source_file"],
        "taxonomy": standardized_filing["taxonomy"],
        "company_name": standardized_filing["filing_metadata"]["company_name"],
        "raw_facts_count": len(standardized_filing["standardized_raw_facts"]),
        "unified_facts_count": len(standardized_filing["standardized_unified_facts"]),
        "contexts_count": len(standardized_filing["contexts"]),
        "filing_metadata_chunks": len(filing_metadata_chunks),
        "raw_fact_chunks": len(raw_fact_chunks),
        "unified_fact_chunks": len(unified_fact_chunks),
        "disclosure_chunks": len(disclosure_chunks),
        "total_chunks": len(all_chunks),
    }

    return {
        "standardized_filing": standardized_filing,
        "chunks": all_chunks,
        "audit": filing_audit,
    }


print("Per-filing chunk builder ready.")

Per-filing chunk builder ready.


## 22. Process All Parser Output JSON Files

In [24]:
structured_chunks: List[Dict[str, Any]] = []
file_audits: List[Dict[str, Any]] = []
failed_files: List[Dict[str, Any]] = []

print(f"Total files found: {len(json_files)}")
print("-" * 80)

for file_path in json_files:
    try:
        parsed_json = load_json(file_path)

        if not isinstance(parsed_json, dict):
            failed_files.append({
                "file": file_path.name,
                "error": f"Expected dict parser output, got {type(parsed_json).__name__}",
            })
            print(f"Skipped {file_path.name}: invalid top-level type {type(parsed_json).__name__}")
            continue

        result = generate_chunks_for_filing(parsed_json, file_path)
        structured_chunks.extend(result["chunks"])
        file_audits.append(result["audit"])

        print(f"Processed: {file_path.name}")
        print(f"  Company       : {result['audit']['company_name']}")
        print(f"  Taxonomy      : {result['audit']['taxonomy']}")
        print(f"  Raw facts     : {result['audit']['raw_facts_count']}")
        print(f"  Unified facts : {result['audit']['unified_facts_count']}")
        print(f"  Chunks        : {result['audit']['total_chunks']}")
        print("-" * 80)

    except Exception as exc:
        failed_files.append({
            "file": file_path.name,
            "error": str(exc),
        })
        print(f"Failed: {file_path.name} -> {exc}")
        print("-" * 80)

print("Completed processing.")
print("Successful files:", len(file_audits))
print("Failed files    :", len(failed_files))
print("Total chunks    :", len(structured_chunks))

Total files found: 9
--------------------------------------------------------------------------------
Processed: GI_117338_1350247_17012025074725.json
  Company       : ICICI Lombard General Insurance Company Limited
  Taxonomy      : INSURANCE
  Raw facts     : 339
  Unified facts : 120
  Chunks        : 273
--------------------------------------------------------------------------------
Processed: INDAS_118123_1365717_30012025030002.json
  Company       : Max Healthcare Institute Limited
  Taxonomy      : COMMERCIAL_NBFC
  Raw facts     : 129
  Unified facts : 10
  Chunks        : 32
--------------------------------------------------------------------------------
rounding
Processed: INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB.json
  Company       : HDFC Bank Limited
  Taxonomy      : BANK
  Raw facts     : 211
  Unified facts : 57
  Chunks        : 124
--------------------------------------------------------------------------------
Processed: LI_117242_1346294_15012025050606

## 23. Preview Sample Structured Chunks

In [25]:
preview_count = min(3, len(structured_chunks))

for i in range(preview_count):
    print(f"Chunk {i + 1}")
    print(json.dumps(structured_chunks[i], indent=2, ensure_ascii=False))
    print("-" * 100)

Chunk 1
{
  "chunk_id": "gi-117338-1350247-17012025074725-f7b38fa1cd4c",
  "text": "Company Name: ICICI Lombard General Insurance Company Limited\nTaxonomy: INSURANCE\nReport Type: Standalone\nReporting Period: Full Filing\nContext Type: Document\nSection: Filing Metadata\n- Company Name: ICICI Lombard General Insurance Company Limited\n- Taxonomy: INSURANCE\n- Report Type: Standalone\n- Presentation Currency: INR\n- Rounding: Unknown Rounding\n- Source File: GI_117338_1350247_17012025074725.xml",
  "metadata": {
    "company_name": "ICICI Lombard General Insurance Company Limited",
    "currency": "INR",
    "rounding": "Unknown Rounding",
    "report_type": "Standalone",
    "source_file": "GI_117338_1350247_17012025074725.xml",
    "taxonomy": "INSURANCE",
    "section_key": "filing_metadata",
    "section_name": "Filing Metadata",
    "chunk_type": "filing_metadata",
    "reporting_period": "Full Filing",
    "context_type": "Document",
    "context_ids": [],
    "start_date": null

## 24. Validate Chunk Schema and Chunk Size Constraints

In [26]:
required_metadata_fields = {
    "company_name", "currency", "rounding", "report_type",
    "source_file", "taxonomy", "section_key", "section_name",
    "chunk_type", "reporting_period", "context_type", "context_ids",
    "source_tags", "unified_fields", "context_signature",
    "chunk_index", "word_count",
}

missing_metadata_chunks = []
empty_text_chunks = []
oversized_chunks = []
duplicate_chunk_ids = []
chunk_ids = []

for chunk in structured_chunks:
    chunk_id = chunk.get("chunk_id")
    chunk_ids.append(chunk_id)

    metadata = chunk.get("metadata", {})
    missing_fields = sorted(field for field in required_metadata_fields if field not in metadata)
    if missing_fields:
        missing_metadata_chunks.append({
            "chunk_id": chunk_id,
            "missing_fields": missing_fields,
        })

    text = chunk.get("text", "")
    if not text or not str(text).strip():
        empty_text_chunks.append(chunk_id)

    if metadata.get("word_count", 0) > max(MAX_WORDS_PER_CHUNK, DISCLOSURE_MAX_WORDS_PER_CHUNK):
        oversized_chunks.append({
            "chunk_id": chunk_id,
            "word_count": metadata.get("word_count", 0),
        })

duplicate_chunk_ids = [chunk_id for chunk_id, count in Counter(chunk_ids).items() if count > 1]

print("Validation summary")
print("-" * 80)
print("Total chunks                  :", len(structured_chunks))
print("Chunks with missing metadata  :", len(missing_metadata_chunks))
print("Chunks with empty text        :", len(empty_text_chunks))
print("Oversized chunks             :", len(oversized_chunks))
print("Duplicate chunk IDs          :", len(duplicate_chunk_ids))

Validation summary
--------------------------------------------------------------------------------
Total chunks                  : 1383
Chunks with missing metadata  : 0
Chunks with empty text        : 0
Oversized chunks             : 0
Duplicate chunk IDs          : 0


## 25. Validate Filing-Level Coverage and Parser Alignment

In [27]:
audit_summary = {
    "files_processed_successfully": len(file_audits),
    "files_failed": len(failed_files),
    "total_chunks": len(structured_chunks),
    "files_with_zero_raw_facts": [],
    "files_with_zero_unified_facts": [],
    "files_with_unknown_company": [],
    "files_with_unknown_taxonomy": [],
    "files_with_zero_chunks": [],
}

for audit in file_audits:
    if audit["raw_facts_count"] == 0:
        audit_summary["files_with_zero_raw_facts"].append(audit["source_file"])
    if audit["unified_facts_count"] == 0:
        audit_summary["files_with_zero_unified_facts"].append(audit["source_file"])
    if str(audit["company_name"]).strip() == "Unknown Company":
        audit_summary["files_with_unknown_company"].append(audit["source_file"])
    if str(audit["taxonomy"]).strip().upper() == "UNKNOWN":
        audit_summary["files_with_unknown_taxonomy"].append(audit["source_file"])
    if audit["total_chunks"] == 0:
        audit_summary["files_with_zero_chunks"].append(audit["source_file"])

print(json.dumps(audit_summary, indent=2, ensure_ascii=False))

{
  "files_processed_successfully": 9,
  "files_failed": 0,
  "total_chunks": 1383,
  "files_with_zero_raw_facts": [],
  "files_with_zero_unified_facts": [],
  "files_with_unknown_company": [],
  "files_with_unknown_taxonomy": [],
  "files_with_zero_chunks": []
}


## 26. Review Validation Exceptions and Failed Files

In [28]:
if missing_metadata_chunks:
    print("Missing metadata examples:")
    for item in missing_metadata_chunks[:5]:
        print(json.dumps(item, indent=2, ensure_ascii=False))
else:
    print("No missing metadata issues found.")

print("-" * 80)

if empty_text_chunks:
    print("Empty text chunk IDs:")
    for item in empty_text_chunks[:5]:
        print(item)
else:
    print("No empty text chunks found.")

print("-" * 80)

if oversized_chunks:
    print("Oversized chunk examples:")
    for item in oversized_chunks[:5]:
        print(json.dumps(item, indent=2, ensure_ascii=False))
else:
    print("No oversized chunks found.")

print("-" * 80)

if duplicate_chunk_ids:
    print("Duplicate chunk IDs:")
    for item in duplicate_chunk_ids[:5]:
        print(item)
else:
    print("No duplicate chunk IDs found.")

print("-" * 80)

if failed_files:
    print("Failed file examples:")
    for item in failed_files[:5]:
        print(json.dumps(item, indent=2, ensure_ascii=False))
else:
    print("No failed files.")

No missing metadata issues found.
--------------------------------------------------------------------------------
No empty text chunks found.
--------------------------------------------------------------------------------
No oversized chunks found.
--------------------------------------------------------------------------------
No duplicate chunk IDs found.
--------------------------------------------------------------------------------
No failed files.


## 27. Inspect Chunk Distribution by Section and Chunk Type

In [29]:
section_counter = Counter(chunk["metadata"]["section_name"] for chunk in structured_chunks)
chunk_type_counter = Counter(chunk["metadata"]["chunk_type"] for chunk in structured_chunks)
taxonomy_counter = Counter(chunk["metadata"]["taxonomy"] for chunk in structured_chunks)

print("Chunk count by section:")
for section_name, count in section_counter.most_common():
    print(f"- {section_name}: {count}")

print("\nChunk count by chunk type:")
for chunk_type, count in chunk_type_counter.most_common():
    print(f"- {chunk_type}: {count}")

print("\nChunk count by taxonomy:")
for taxonomy, count in taxonomy_counter.most_common():
    print(f"- {taxonomy}: {count}")

Chunk count by section:
- Unified Financial Summary: 439
- Segment Performance: 395
- Income Statement & Profitability: 216
- Balance Sheet – Assets: 125
- Narrative Notes & Disclosures: 94
- Other Comprehensive Income (OCI): 33
- Reporting Metadata & Governance: 23
- Capital, EPS & Ratios: 21
- Cash Flow Reconciliation Adjustments: 15
- Filing Metadata: 9
- Balance Sheet – Equity & Liabilities: 7
- Cash Flow Statement: 6

Chunk count by chunk type:
- raw_fact_group: 723
- unified_metric_group: 557
- narrative_disclosure: 94
- filing_metadata: 9

Chunk count by taxonomy:
- INSURANCE: 668
- BANK: 381
- COMMERCIAL_NBFC: 334


## 28. Build Final Audit Payload

In [30]:
final_audit_payload = {
    "pipeline_version": "financial_structured_chunks_v2",
    "input_folder": str(INPUT_JSON_FOLDER.resolve()),
    "output_file": str(OUTPUT_JSON_PATH.resolve()),
    "files_discovered": len(json_files),
    "files_processed_successfully": len(file_audits),
    "files_failed": len(failed_files),
    "total_chunks": len(structured_chunks),
    "validation": {
        "missing_metadata_chunks": len(missing_metadata_chunks),
        "empty_text_chunks": len(empty_text_chunks),
        "oversized_chunks": len(oversized_chunks),
        "duplicate_chunk_ids": len(duplicate_chunk_ids),
    },
    "coverage": audit_summary,
    "chunk_distribution": {
        "by_section": dict(section_counter),
        "by_chunk_type": dict(chunk_type_counter),
        "by_taxonomy": dict(taxonomy_counter),
    },
    "file_audits": file_audits,
    "failed_files": failed_files,
}

print(json.dumps({
    "files_processed_successfully": final_audit_payload["files_processed_successfully"],
    "files_failed": final_audit_payload["files_failed"],
    "total_chunks": final_audit_payload["total_chunks"],
    "validation": final_audit_payload["validation"],
}, indent=2, ensure_ascii=False))

{
  "files_processed_successfully": 9,
  "files_failed": 0,
  "total_chunks": 1383,
  "validation": {
    "missing_metadata_chunks": 0,
    "empty_text_chunks": 0,
    "oversized_chunks": 0,
    "duplicate_chunk_ids": 0
  }
}


## 29. Save Structured Chunk Records and Audit Output

In [31]:
output_path = save_json(structured_chunks, OUTPUT_JSON_PATH)
audit_path = save_json(final_audit_payload, AUDIT_JSON_PATH)

print("Structured chunks saved successfully.")
print("Chunks output path:", output_path.resolve())
print("Chunks file size (KB):", round(output_path.stat().st_size / 1024, 2))
print("Audit output path :", audit_path.resolve())
print("Audit file size (KB):", round(audit_path.stat().st_size / 1024, 2))

Structured chunks saved successfully.
Chunks output path: C:\Users\Home\llmops-nse-rag\data\processed\chunks\financial_structured_chunks_v2.json
Chunks file size (KB): 2265.16
Audit output path : C:\Users\Home\llmops-nse-rag\data\processed\audits\financial_structured_chunks_audit_v2.json
Audit file size (KB): 5.37


## 30. Final Output Summary

In [32]:
print(f"Total files discovered            : {len(json_files)}")
print(f"Total files processed successfully: {len(file_audits)}")
print(f"Total failed files                : {len(failed_files)}")
print(f"Total structured chunks created   : {len(structured_chunks)}")
print(f"Chunks output file                : {output_path.name}")
print(f"Audit output file                 : {audit_path.name}")
print(f"Chunks file size (KB)             : {round(output_path.stat().st_size / 1024, 2)}")
print(f"Audit file size (KB)              : {round(audit_path.stat().st_size / 1024, 2)}")

Total files discovered            : 9
Total files processed successfully: 9
Total failed files                : 0
Total structured chunks created   : 1383
Chunks output file                : financial_structured_chunks_v2.json
Audit output file                 : financial_structured_chunks_audit_v2.json
Chunks file size (KB)             : 2265.16
Audit file size (KB)              : 5.37
